# Phase 2: Recommender Model Benchmarking with Cornac

This notebook evaluates multiple recommendation algorithms (Popularity Baseline, Neighborhood CF, Matrix Factorization, and SVD) across multiple datasets (MovieLens-1M and FilmTrust) using the **Cornac** framework.

We evaluate models on both **Ranking Accuracy** (`NDCG@10`, `Recall@10`, `Precision@10`, `MAP`, `MRR`) and **Popularity Bias / Item Exposure Inequality** (`Gini@10`).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import cornac
from cornac.datasets import movielens, filmtrust
from cornac.eval_methods import RatioSplit
from cornac.models import MostPop, UserKNN, ItemKNN, BPR, PMF, MF, SVD
from cornac.metrics import NDCG, Recall, Precision, MAP, MRR

print(f"Cornac Version: {cornac.__version__}")

## 1. Load Datasets

We load two implicit feedback datasets:
1. **MovieLens 1M**: ~1 million interaction records from 6,038 users on 3,533 items.
2. **FilmTrust**: ~35,497 interaction records from 1,508 users on 2,071 items.

In [ ]:
# Load MovieLens 1M tuple interactions (user_id, item_id, rating)
ml_1m_data = movielens.load_feedback(variant="1M")
print(f"MovieLens 1M interactions count: {len(ml_1m_data):,}")

# Load FilmTrust tuple interactions
filmtrust_data = filmtrust.load_feedback()
print(f"FilmTrust interactions count: {len(filmtrust_data):,}")

## 2. Define Train/Test Evaluation Splits

We use an **80/20 Train/Test RatioSplit** per user, filtering positive interactions with `rating_threshold = 4.0` for implicit binarization.

In [ ]:
SEED = 42

# MovieLens 1M Evaluation Split
eval_ml1m = RatioSplit(
    data=ml_1m_data,
    test_size=0.2,
    rating_threshold=4.0,
    seed=SEED,
    exclude_unknowns=True,
    verbose=True
)

# FilmTrust Evaluation Split
eval_filmtrust = RatioSplit(
    data=filmtrust_data,
    test_size=0.2,
    rating_threshold=3.5,  # FilmTrust ratings range 0.5 - 4.0
    seed=SEED,
    exclude_unknowns=True,
    verbose=True
)

## 3. Instantiate Recommendation Models

We instantiate 7 native models covering diverse algorithmic paradigms:
1. `MostPop`: Popularity-based baseline (max popularity bias).
2. `UserKNN`: User-based Collaborative Filtering with Cosine similarity.
3. `ItemKNN`: Item-based Collaborative Filtering with Cosine similarity.
4. `BPR`: Bayesian Personalized Ranking matrix factorization.
5. `PMF`: Probabilistic Matrix Factorization.
6. `MF`: Matrix Factorization.
7. `SVD`: Singular Value Decomposition recommendation model.

In [ ]:
models = [
    MostPop(name="MostPop"),
    UserKNN(k=20, similarity="cosine", name="UserKNN"),
    ItemKNN(k=20, similarity="cosine", name="ItemKNN"),
    BPR(k=20, max_iter=100, learning_rate=0.01, lambda_reg=0.01, seed=SEED, name="BPR"),
    PMF(k=20, max_iter=100, learning_rate=0.001, lambda_reg=0.01, seed=SEED, name="PMF"),
    MF(k=20, max_iter=100, learning_rate=0.01, lambda_reg=0.01, seed=SEED, name="MF"),
    SVD(k=20, max_iter=100, learning_rate=0.01, lambda_reg=0.01, seed=SEED, name="SVD")
]

## 4. Define Evaluation Metrics & Popularity Bias Gini Function

We evaluate top-10 recommendations using both Ranking Accuracy & Gini Exposure Inequality:

In [ ]:
metrics = [
    NDCG(k=10),
    Recall(k=10),
    Precision(k=10),
    MAP(),
    MRR()
]

def compute_gini(exposure_counts):
    x = np.sort(exposure_counts)
    n = len(x)
    cum = np.cumsum(x)
    if cum[-1] == 0:
        return 0.0
    return (n + 1 - 2 * np.sum(cum) / cum[-1]) / n

def calculate_model_gini(model, eval_method, top_k=10):
    test_users = list(eval_method.test_set.user_data.keys())
    num_items = eval_method.total_items
    item_counts = np.zeros(num_items)
    
    for uid in test_users:
        recs = model.recommend(user_id=uid, k=top_k, remove_seen=True)
        if recs is not None:
            for item_idx in recs:
                item_counts[item_idx] += 1
                
    return compute_gini(item_counts)

## 5. Run Benchmark Experiment on MovieLens 1M

In [ ]:
print("=== Running Experiment on MovieLens 1M ===")
exp_ml1m = cornac.Experiment(
    eval_method=eval_ml1m,
    models=models,
    metrics=metrics,
    user_based=True
)
exp_ml1m.run()

## 6. Run Benchmark Experiment on FilmTrust

In [ ]:
print("=== Running Experiment on FilmTrust ===")
exp_filmtrust = cornac.Experiment(
    eval_method=eval_filmtrust,
    models=models,
    metrics=metrics,
    user_based=True
)
exp_filmtrust.run()

## 7. Comparative Analysis & Visualization (Accuracy vs Popularity Bias)

We extract metrics and compute the Gini exposure inequality for each model across both datasets.

In [ ]:
def extract_experiment_df(exp, eval_method, dataset_name):
    rows = []
    for res in exp.result:
        row = {"Dataset": dataset_name, "Model": res.model_name}
        for metric, val in zip(exp.metrics, res.metric_avg):
            row[metric.name] = val
        # Compute Gini Popularity Bias
        row["Gini@10"] = calculate_model_gini(res.model, eval_method, top_k=10)
        rows.append(row)
    return pd.DataFrame(rows)

df_ml1m = extract_experiment_df(exp_ml1m, eval_ml1m, "MovieLens 1M")
df_filmtrust = extract_experiment_df(exp_filmtrust, eval_filmtrust, "FilmTrust")

df_all = pd.concat([df_ml1m, df_filmtrust], ignore_index=True)
print(df_all.to_string(index=False))

# Plot NDCG@10 and Gini@10 Comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# MovieLens 1M NDCG
sns.barplot(data=df_ml1m, x="Model", y="NDCG@10", ax=axes[0, 0], palette="Blues_d")
axes[0, 0].set_title("MovieLens 1M - NDCG@10 (Accuracy)")
axes[0, 0].tick_params(axis='x', rotation=30)
axes[0, 0].grid(True, linestyle="--", alpha=0.5)

# MovieLens 1M Gini
sns.barplot(data=df_ml1m, x="Model", y="Gini@10", ax=axes[0, 1], palette="Purples_d")
axes[0, 1].set_title("MovieLens 1M - Gini@10 (Popularity Bias)")
axes[0, 1].tick_params(axis='x', rotation=30)
axes[0, 1].grid(True, linestyle="--", alpha=0.5)

# FilmTrust NDCG
sns.barplot(data=df_filmtrust, x="Model", y="NDCG@10", ax=axes[1, 0], palette="Oranges_d")
axes[1, 0].set_title("FilmTrust - NDCG@10 (Accuracy)")
axes[1, 0].tick_params(axis='x', rotation=30)
axes[1, 0].grid(True, linestyle="--", alpha=0.5)

# FilmTrust Gini
sns.barplot(data=df_filmtrust, x="Model", y="Gini@10", ax=axes[1, 1], palette="Reds_d")
axes[1, 1].set_title("FilmTrust - Gini@10 (Popularity Bias)")
axes[1, 1].tick_params(axis='x', rotation=30)
axes[1, 1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()